In [1]:
import json
import glob
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display


RESULTS_BASE = Path('/home/weiyong/repos/MADE/results/baselines/custom/qwen3dot5-122b-reflx-tracking/systems_ternary_n10_maxatoms20_intermetallic_smact/20260317-220021/llm_react_orchestrator_systems_ternary_n10_maxatoms20_intermetallic_smact_2systems_50queries_100stabilitymeV')
SYSTEM = "Au-K-Tb"

ELEMENTS = SYSTEM.split('-')
print(ELEMENTS)
TRACE_DIR = RESULTS_BASE / "llm_traces" / SYSTEM

# first load all episodes 
episodes = {} 
for ep_file in sorted(TRACE_DIR.glob("episode_*.jsonl")):
    ep_idx = int(ep_file.stem.split("_")[1])
    with open(ep_file) as f: 
        episodes[ep_idx] = [json.loads(line) for line in f ]


['Au', 'K', 'Tb']


In [2]:
episodes[2][49]['extra']['behavior_metrics']

{'selection_total': 50.0,
 'selection_unique_count': 7.0,
 'selection_new_composition_count': 7.0,
 'selection_new_composition_rate': 0.14,
 'selection_top1_share': 0.22,
 'selection_entropy': 1.8826520517813847,
 'selection_entropy_normalized': 0.9674917686694637,
 'selection_switch_rate': 0.8775510204081632,
 'selection_max_same_composition_streak': 2.0,
 'targeted_unique_count': 7.0,
 'targeted_new_total': 7.0,
 'targeted_revisit_total': 170.0,
 'explore_vs_exploit_ratio': 0.041176470588235294,
 'tool_arg_element_calls_total': 346.0,
 'element_arg_count_Au': 346.0,
 'element_arg_presence_ratio_Au': 1.0,
 'element_arg_count_K': 0.0,
 'element_arg_presence_ratio_K': 0.0,
 'element_arg_count_Tb': 346.0,
 'element_arg_presence_ratio_Tb': 1.0,
 'buffer_churn_generated_total': 368.0,
 'buffer_churn_added_total': 306.0,
 'buffer_churn_duplicate_total': 62.0,
 'buffer_churn_cached_total': 0.0,
 'buffer_churn_static_filtered_total': 0.0,
 'buffer_churn_created_added_total': 0.0,
 'buffer_chu

### Helper Functions for Analysis


In [5]:
def extract_reflection(episodes, ep_idx): 

    steps = episodes[ep_idx]
    for step in steps:
        if step.get("component") == "self_reflection":
            reflection = step.get("output", {}).get("reflection", "(empty)")
            return reflection 

                
def extract_selected_compositions(steps):
    #Extract all compositions selected for evaluation in an episode.
    selected = []
    for step in steps:
        if step.get("component") != "orchestrator":
            continue
        traj = step.get("output", {}).get("trajectory", {})
        for k, v in sorted(traj.items()):
            if isinstance(v, str) and "SELECTED:" in v:
                parts = v.split("reduced:")
                if len(parts) > 1:
                    comp = parts[1].split(",")[0].strip()
                    selected.append(comp)
    return selected

    
for ep_idx, steps in episodes.items(): 
    
    display(Markdown(f"### Episode {ep_idx}: \n {extract_reflection(episodes, ep_idx)}")) 
    print('='*80)

### Episode 0: 
 *   **Prioritize Au-Tb Binary Subsystem:** Focus search efforts on the Au-Tb edge first, as K-containing ternaries showed consistently high instability (>0.25 eV/atom).
*   **Restrict Potassium Content:** High K stoichiometry appears thermodynamically unfavorable; limit K concentration to <20 at% to reduce risk of phase separation.
*   **Cap Redundant Stoichiometry Queries:** Limit sampling of any single ternary formula to 2 attempts; if energies remain >0.2 eV/atom, switch to new compositions immediately.
*   **Prefer Smaller Unit Cells:** Bias structure generation toward cells with <6 sites, as the lowest energy structure (Au2Tb1) was a 3-site cell.

### Episode 1: 
 *   **Eliminate Potassium:** Completely exclude K-containing compositions from the search space; all K-ternaries evaluated were highly unstable (>0.15 eV/atom).
*   **Enforce Stoichiometry Deduplication:** Implement a strict check to prevent re-querying identical stoichiometries (e.g., Au3Tb2 was queried twice with no improvement).
*   **Target Specific Ratios:** Narrow the focus to Au-Tb stoichiometries near 3:2 and 3:1 where the lowest energies (0.029-0.041 eV/atom) were observed, rather than broad exploration.
*   **Bias Towards Small Cells:** Prioritize structure generation for unit cells with 4-6 sites, as these correlated with the most stable candidates found.
*   **Prioritize Known Phases:** Shift strategy from novelty to recall; explicitly query known stable stoichiometries from literature to recover the 0.000 recall score.

### Episode 2: 
 on ## ]]
        {reflection}

### Episode 3: 
 *   **Enforce Stoichiometry Uniqueness:** Implement a hard limit of one query per unique stoichiometry (e.g., Au5Tb1). If the first structure for a ratio exceeds 0.05 eV/atom, discard the ratio immediately rather than retrying with different structures.
*   **Prioritize Known Phases for Recall:** Dedicate the first 20% of the query budget to explicitly generating and evaluating known Au-Tb intermetallics from literature (e.g., AuTb, Au3Tb) to guarantee a non-zero recall score before exploring novel compositions.
*   **Restrict Unit Cell Size:** Limit structure generation to cells with ≤ 8 sites. Large cells (12-17 sites) previously consumed significant resources without yielding confirmed stable phases and contradict established stability trends.
*   **Target Stricter Energy Thresholds:** Treat structures with e_above_hull > 0.02 eV/atom as unstable for the purpose of stopping conditions. The episode produced several "stable" labeled structures (0.03-0.09 eV/atom) that were not counted as successful discoveries.

### Episode 4: 
 *   **Fix Stability Counting Threshold**: Ensure structures with e_above_hull < 0.05 eV/atom are counted as stable; the episode found 3 such structures (#3, #11, #15) but outcome reported 0. Verify oracle threshold alignment.
*   **Limit Stoichiometry Redundancy**: Cap queries per unique stoichiometry at 3 attempts. If all exceed 0.1 eV/atom, abandon that ratio rather than continuing (e.g., Au2Tb1 was queried 6 times with 1 success).
*   **Prioritize Proven Ratios**: Concentrate 60%+ of queries on Au4Tb1 and Au2Tb1 ratios which produced stable structures, rather than exploring low-yield compositions like Au1Tb2 (all >0.17 eV/atom).
*   **Restrict Unit Cell Size**: Limit generation to ≤6 sites; 8-9 site cells (#10) consumed resources without yielding stable phases. Successful structures (#3, #11, #15) had 5-6 sites.
*   **Maintain K-Exclusion**: Confirm K is completely removed from search space; prior episodes showed all K-ternaries were highly unstable (>0.15 eV/atom).

### analysis of number of times an element is selected for evaluation:

In [8]:

print("=" * 80)
print(f"{'Episode':<10} {'Total':<8} {'Au-containing':<15} {'K-containing':<15} {'Tb-containing': <15} {'Unique comps':<15}")
print("=" * 80)

ep_selections = {}
for ep_idx, steps in episodes.items():
    comps = extract_selected_compositions(steps)
    ep_selections[ep_idx] = comps



    selected_elem_counts = [0,0,0]
    # count number of times each element shows up in selection 
    for idx, element in enumerate(ELEMENTS): 
        elem_count = sum(1 for c in comps if element in c)
        selected_elem_counts[idx] = elem_count

    #print(selected_elem_counts)
    unique = len(set(comps))
    print(f"{ep_idx:<10} {len(comps):<8} {selected_elem_counts[0]:<15} {selected_elem_counts[1]:<15} {selected_elem_counts[2]:<15} {unique:<15}")

Episode    Total    Au-containing   K-containing    Tb-containing   Unique comps   
0          158      158             107             131             14             
1          139      139             21              137             28             
2          150      150             0               150             7              
3          139      139             0               139             15             
4          143      143             0               143             7              


### Composition selection across episodes

In [11]:
for ep_idx in sorted(ep_selections.keys()):
    comps = ep_selections[ep_idx]
    counter = Counter(comps)
    print(f"\n--- Episode {ep_idx} ({len(comps)} selections) ---")
    for comp, count in counter.most_common(10):
        print(f"  {comp:<15} {count:>3}")


--- Episode 0 (158 selections) ---
  KTbAu            21
  K2Tb2Au          21
  K2Au3            16
  TbAu             14
  TbAu2            13
  K2TbAu           13
  Tb2Au            13
  KTb2Au           13
  TbAu3            11
  KAu5             11

--- Episode 1 (139 selections) ---
  TbAu             16
  Tb2Au3           15
  TbAu3            15
  TbAu5            12
  Tb2Au            11
  Tb3Au2           11
  Tb3Au             9
  TbAu2             6
  Tb2Au5            6
  TbAu4             5

--- Episode 2 (150 selections) ---
  Tb2Au            33
  TbAu             31
  TbAu3            21
  Tb3Au            19
  TbAu2            17
  Tb3Au2           16
  Tb2Au3           13

--- Episode 3 (139 selections) ---
  TbAu3            28
  Tb2Au3           18
  TbAu             16
  TbAu4            14
  TbAu2            11
  Tb2Au            10
  TbAu5             9
  Tb3Au2            8
  Tb3Au             6
  Tb2Au5            5

--- Episode 4 (143 selections) ---
  TbAu

### Tracking number of new families introduced
new family: different reduced composition

In [14]:
# Track: when does the agent stop generating new families within each episode?

# store {(ep, step, num)} Tuple for number of families added. 

families = []


for ep_idx, steps in episodes.items():
    new_fam_steps = []
    for i, step in enumerate(steps):
        if step.get("component") != "orchestrator":
            continue
        bm = step.get("extra", {}).get("behavior_metrics", {})
        nf = bm.get("num_new_families", 0)
        if nf > 0:
            new_fam_steps.append((i, int(nf)))
    
    total_steps = sum(1 for s in steps if s.get("component") == "orchestrator")
    last_new = new_fam_steps[-1][0] if new_fam_steps else -1
    total_new = sum(n for _, n in new_fam_steps)
    print(f"Episode {ep_idx}: {total_new} new families, last new family at step {last_new}/{total_steps-1}")
    if new_fam_steps:
        for s, n in new_fam_steps:
            print(f"  Step {s}: +{n} new families")
            families.append((ep_idx, s, n))



Episode 0: 14 new families, last new family at step 1/49
  Step 0: +9 new families
  Step 1: +5 new families
Episode 1: 38 new families, last new family at step 45/49
  Step 0: +9 new families
  Step 1: +1 new families
  Step 2: +5 new families
  Step 16: +1 new families
  Step 25: +7 new families
  Step 34: +3 new families
  Step 37: +3 new families
  Step 38: +6 new families
  Step 39: +1 new families
  Step 45: +2 new families
Episode 2: 7 new families, last new family at step 17/49
  Step 0: +6 new families
  Step 17: +1 new families
Episode 3: 30 new families, last new family at step 47/49
  Step 0: +2 new families
  Step 1: +1 new families
  Step 2: +2 new families
  Step 5: +1 new families
  Step 6: +4 new families
  Step 8: +2 new families
  Step 11: +1 new families
  Step 17: +3 new families
  Step 19: +4 new families
  Step 27: +5 new families
  Step 33: +1 new families
  Step 35: +2 new families
  Step 39: +1 new families
  Step 47: +1 new families
Episode 4: 12 new families

### Track the ReAct chain during the steps which have the most new families. 